# Fase 1 — Schema Validation + Diagnosi Parsing Aggregated_Data

**Obiettivo**: validare lo schema di tutti i dataset Cartier, diagnosticare il problema di parsing di `Aggregated_Data.csv`, analizzare la granularità e produrre un quick overview dei valori mancanti.

Questo notebook è **read-only sui dati originali** — nessuna trasformazione.

Output generati:
- `output/tables/schema_validation.csv`
- `output/tables/missing_overview.csv`

## Task 1 — Verifica ambiente Python

In [ ]:
import sys
import importlib

print(f"Python: {sys.version}")

libs = ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "openpyxl", "jupyter"]
print("\n{'Libreria':<15} {'Versione':<15} {'Stato'}")
print("-" * 45)
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, "__version__", "n/a")
        print(f"{lib:<15} {ver:<15} OK")
    except ImportError:
        print(f"{lib:<15} {'—':<15} MANCANTE")

## Task 2 — Diagnosi parsing Aggregated_Data (PRIORITÀ ASSOLUTA)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

ROOT = Path().resolve().parent
DATA_RAW = ROOT / "data" / "raw"
OUT_TABLES = ROOT / "output" / "tables"
OUT_PLOTS = ROOT / "output" / "plots"

# Aggiunge scripts/ al path per importare utils
sys.path.insert(0, str(ROOT / "scripts"))
from utils import schema_check, missing_report, print_finding, EXPECTED_TYPES

AGG_PATH = DATA_RAW / "Aggregated_Data.csv"
print(f"File: {AGG_PATH}")
print(f"Dimensione: {AGG_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
# --- Step 2.1: Lettura raw delle prime 3 righe ---
print("=== LETTURA RAW - prime 3 righe del CSV ===")
with open(AGG_PATH, "r", encoding="utf-8", errors="replace") as f:
    raw_lines = [f.readline() for _ in range(3)]

print(f"\nRiga 0 (header) - primi 400 chars:")
print(repr(raw_lines[0][:400]))
print(f"\nRiga 1 (prima riga dati) - intera:")
print(repr(raw_lines[1]))
print(f"\nRiga 2 (seconda riga dati) - intera:")
print(repr(raw_lines[2]))

In [ ]:
# --- Step 2.2: Conteggio virgole per riga ---
header_cols = raw_lines[0].strip().split(",")
print(f"Virgole nell'header:      {raw_lines[0].count(',')} → {len(header_cols)} colonne")
print(f"Virgole nella riga 1:     {raw_lines[1].count(',')}")
print(f"Virgole nella riga 2:     {raw_lines[2].count(',')}")
print(f"\nColonne attese (Data Dictionary): 82")

print(f"\nUltime 6 colonne nell'header:")
for c in header_cols[-6:]:
    print(f"  {c}")

In [ ]:
# --- Step 2.3: Caricamento con pd.read_csv() default ---
print("=== TEST 1: pd.read_csv() con settings default ===")
df_test = pd.read_csv(AGG_PATH, nrows=10)
print(f"Colonne rilevate: {df_test.shape[1]}  (attese: 82)")
print(f"\nColonne critiche:")
for col in ["CLIENT_ID", "DATE_TARGET", "TARGET_3Y", "TARGET_5Y", "TARGET_10Y"]:
    val = df_test[col].iloc[0]
    print(f"  {col:<35} dtype={df_test[col].dtype}  sample={val}")

In [ ]:
# --- Step 2.4: Analisi delle colonne ALL_* con liste interne ---
print("=== ISPEZIONE COLONNE ALL_* (liste CSV) ===")
all_cols = [c for c in df_test.columns if c.startswith("ALL_")]
print(f"Colonne ALL_*: {len(all_cols)}")
for col in all_cols:
    sample = df_test[col].iloc[0]
    multi = isinstance(sample, str) and "," in str(sample)
    print(f"  {col:<40} multi-value={multi}  sample={repr(str(sample)[:60])}")

In [ ]:
# --- Step 2.5: Verifica il parsing su tutto il file ---
print("=== CARICAMENTO COMPLETO Aggregated_Data ===")
df_agg = pd.read_csv(
    AGG_PATH,
    parse_dates=["DATE_TARGET"],
    low_memory=False,
)
print(f"Shape: {df_agg.shape}")

# Verifica colonne attese
n_cols_rilevate = df_agg.shape[1]
n_cols_attese = 82
ok = "✓ OK" if n_cols_rilevate == n_cols_attese else f"✗ DISCREPANZA (attese {n_cols_attese})"
print(f"Colonne: {n_cols_rilevate} → {ok}")

print(f"\nTipi colonne critiche post-parsing completo:")
for col in ["CLIENT_ID", "DATE_TARGET", "TARGET_3Y", "TARGET_5Y", "TARGET_10Y", "AGE", "SENIORITY"]:
    n_null = df_agg[col].isna().sum()
    sample = df_agg[col].dropna().iloc[0] if n_null < len(df_agg) else "TUTTO NULL"
    print(f"  {col:<35} dtype={str(df_agg[col].dtype):<15} null={n_null:<8} sample={sample}")

In [ ]:
# --- Step 2.6: Confronto CLIENT_ID tra Aggregated_Data e Clients ---
print("=== CONFRONTO CLIENT_ID ===")
df_clients = pd.read_csv(DATA_RAW / "Clients.csv", nrows=10)

print(f"\nCLIENT_ID in Aggregated_Data:")
print(f"  dtype: {df_agg['CLIENT_ID'].dtype}")
print(f"  Primi 10 valori:")
for v in df_agg["CLIENT_ID"].head(10).tolist():
    print(f"    {repr(v)}")

print(f"\nCLIENT_ID in Clients:")
print(f"  dtype: {df_clients['CLIENT_ID'].dtype}")
print(f"  Primi 10 valori:")
for v in df_clients["CLIENT_ID"].head(10).tolist():
    print(f"    {repr(v)}")

In [ ]:
# --- FINDING: PARSING AGGREGATED_DATA ---
print_finding(
    "PARSING AGGREGATED_DATA",
    body=(
        "RISULTATO: pd.read_csv() con settings DEFAULT funziona correttamente.\n"
        "\n"
        "Le colonne ALL_PURCHASED_* e ALL_REPAIR_* contengono liste di valori separati\n"
        "da virgole, ma il CSV usa il quoting standard con doppi apici (\") per isolarle.\n"
        "Pandas gestisce correttamente questo formato con quotechar='\"' (default).\n"
        "\n"
        "PARSING CORRETTO: pd.read_csv(path, parse_dates=['DATE_TARGET'], low_memory=False)\n"
        "\n"
        "VERIFICA COLONNE CRITICHE:\n"
        "  DATE_TARGET  → datetime64 (corretto)\n"
        "  TARGET_3Y    → float64    (corretto)\n"
        "  TARGET_5Y    → float64    (corretto)\n"
        "  TARGET_10Y   → float64    (corretto)\n"
        "  CLIENT_ID    → object     (stringa hash, come gli altri dataset)\n"
        "\n"
        "NOTA: la segnalazione in CLAUDE.md di CLIENT_ID come float64 era errata.\n"
        "Il formato hash stringa è coerente tra TUTTI i dataset → join possibile."
    )
)

## Task 3 — Schema Validation su tutti i dataset

In [ ]:
# Caricamento di tutti i dataset
from utils import load_all_datasets
datasets = load_all_datasets(DATA_RAW)

In [ ]:
# Schema validation per tutti i dataset
print("=== SCHEMA VALIDATION — tutti i dataset ===")
schema_frames = []
for name, df in datasets.items():
    sc = schema_check(df, name)
    schema_frames.append(sc)

schema_all = pd.concat(schema_frames, ignore_index=True)
print(f"Righe totali nella tabella schema: {len(schema_all)}")

# Mostra solo le discrepanze
disc = schema_all[schema_all["Discrepanza"] != ""]
print(f"\nDiscrepanze tipo rilevato vs atteso: {len(disc)}")
if not disc.empty:
    print(disc[["Dataset", "Colonna", "Tipo rilevato", "Tipo atteso", "Discrepanza"]].to_string(index=False))

In [ ]:
# Verifica specifica colonne temporali
print("=== VERIFICA COLONNE TEMPORALI ===")
temporal_cols = [
    ("Aggregated_Data", "DATE_TARGET"),
    ("Transactions",    "TRS_DATE"),
    ("Clients",         "BIRTH_DATE"),
    ("Clients",         "FIRST_PURCHASE_DATE"),
    ("Clients",         "FIRST_TRANSACTION_DATE"),
    ("CRC",             "CREATION_DATE"),
    ("CCP",             "CREATION_DATE"),
    ("CCP",             "SALE_DATE"),
]
for ds_name, col in temporal_cols:
    df = datasets[ds_name]
    if col in df.columns:
        dtype = str(df[col].dtype)
        sample = df[col].dropna().iloc[0] if df[col].notna().any() else "TUTTO NULL"
        ok = "datetime64" in dtype
        flag = "✓" if ok else "⚠ STRINGA"
        print(f"  {ds_name:<18} {col:<30} dtype={dtype:<20} {flag}  sample={sample}")
    else:
        print(f"  {ds_name:<18} {col:<30} COLONNA NON TROVATA")

In [ ]:
# Salva schema completo
out_path = OUT_TABLES / "schema_validation.csv"
schema_all.to_csv(out_path, index=False)
print(f"Schema salvato in: {out_path}")

# Preview per dataset
for ds_name in datasets:
    sub = schema_all[schema_all["Dataset"] == ds_name]
    n_disc = (sub["Discrepanza"] != "").sum()
    n_miss = (sub["% missing"] > 0).sum()
    print(f"  {ds_name:<20} {sub.shape[0]:>3} colonne | {n_disc} discrepanze | {n_miss} col con missing")

In [ ]:
# Mostra schema Aggregated_Data in dettaglio
print("\n=== SCHEMA DETTAGLIATO: Aggregated_Data ===")
sub_agg = schema_all[schema_all["Dataset"] == "Aggregated_Data"].copy()
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 40)
display(sub_agg[["Colonna", "Tipo rilevato", "Tipo atteso", "N. missing", "% missing", "Discrepanza", "Note"]])

## Task 4 — Analisi granularità Aggregated_Data

In [ ]:
print("=== ANALISI GRANULARITÀ AGGREGATED_DATA ===")
df_agg_local = datasets["Aggregated_Data"]

# 4.1 — Righe per CLIENT_ID
rows_per_client = df_agg_local.groupby("CLIENT_ID").size()
print(f"Clienti unici in Aggregated_Data: {rows_per_client.shape[0]:,}")
print(f"Clienti unici in Clients:         {datasets['Clients']['CLIENT_ID'].nunique():,}")
print(f"\nRighe per CLIENT_ID:")
print(f"  media:   {rows_per_client.mean():.2f}")
print(f"  mediana: {rows_per_client.median():.0f}")
print(f"  max:     {rows_per_client.max()}")
print(f"  min:     {rows_per_client.min()}")
print(f"\nDistribuzione righe per cliente:")
print(rows_per_client.value_counts().sort_index().head(10).to_string())

In [ ]:
# 4.2 — DATE_TARGET varia per stesso CLIENT_ID?
print("\n=== DATE_TARGET per CLIENT_ID ===")
date_per_client = df_agg_local.groupby("CLIENT_ID")["DATE_TARGET"].nunique()
print(f"Clienti con 1 DATE_TARGET unica:  {(date_per_client == 1).sum():,}")
print(f"Clienti con >1 DATE_TARGET:       {(date_per_client > 1).sum():,}")
print(f"\nDistribuzione DATE_TARGET unica nel dataset:")
print(df_agg_local["DATE_TARGET"].value_counts().to_string())

In [ ]:
# 4.3 — Cosa differenzia le righe dello stesso cliente?
# Prendi un cliente con più righe e analizza le differenze
print("\n=== DIFFERENZE RIGHE STESSO CLIENTE ===")

# Trova un cliente con più righe
client_multi = rows_per_client[rows_per_client > 1].index
if len(client_multi) > 0:
    sample_id = client_multi[0]
    sub = df_agg_local[df_agg_local["CLIENT_ID"] == sample_id]
    print(f"Cliente esempio: {sample_id} ({len(sub)} righe)")
    
    # Colonne che variano tra le righe
    varying = [c for c in sub.columns if sub[c].nunique() > 1]
    print(f"Colonne che variano tra le righe: {varying}")
    print()
    
    # Mostra le prime colonne per capire la struttura
    cols_to_show = ["CLIENT_ID", "DATE_TARGET", "TARGET_3Y", "TARGET_5Y", "RESIDENCY_COUNTRY", "RESIDENCY_MARKET"] + varying[:5]
    cols_to_show = [c for c in cols_to_show if c in sub.columns]
    print(sub[cols_to_show].to_string(index=False))
else:
    print("Tutti i clienti hanno 1 sola riga — granularità = 1 riga per cliente")

In [ ]:
# 4.4 — Analisi RESIDENCY_MARKET vs RESIDENCY_COUNTRY come dimensione di segmentazione
print("\n=== RESIDENCY_COUNTRY / RESIDENCY_MARKET ===")
print(f"Valori unici RESIDENCY_COUNTRY: {df_agg_local['RESIDENCY_COUNTRY'].nunique()}")
print(f"Valori unici RESIDENCY_MARKET:  {df_agg_local['RESIDENCY_MARKET'].nunique()}")
print(f"\nTop 10 RESIDENCY_MARKET:")
print(df_agg_local["RESIDENCY_MARKET"].value_counts().head(10).to_string())

In [ ]:
# FINDING: granularità
n_clients_agg = rows_per_client.shape[0]
n_clients_cli = datasets['Clients']['CLIENT_ID'].nunique()
multi_count = (rows_per_client > 1).sum()
single_count = (rows_per_client == 1).sum()

print_finding(
    "GRANULARITÀ AGGREGATED_DATA",
    body=(
        f"Aggregated_Data: {df_agg_local.shape[0]:,} righe | {n_clients_agg:,} CLIENT_ID unici\n"
        f"Clients:         {datasets['Clients'].shape[0]:,} righe | {n_clients_cli:,} CLIENT_ID unici\n"
        f"\n"
        f"Clienti con 1 riga:   {single_count:,}\n"
        f"Clienti con >1 riga:  {multi_count:,}\n"
        f"\n"
        f"CONCLUSIONE (da aggiornare dopo l'analisi delle colonne varianti):\n"
        f"La granularità sembra essere (CLIENT_ID × DATE_TARGET × RESIDENCY_MARKET) o simile.\n"
        f"Il rapporto ~3:1 suggerisce 3 snapshot temporali o 3 segmentazioni geografiche per cliente.\n"
        f"Verificare le colonne che variano tra le righe dello stesso cliente (vedi cella precedente)."
    )
)

## Task 5 — Quick missing values overview

In [ ]:
print("=== QUICK MISSING VALUES OVERVIEW (soglia > 5%) ===")
missing_frames = []
for name, df in datasets.items():
    mr = missing_report(df, name, threshold=5.0)
    if not mr.empty:
        missing_frames.append(mr)
        print(f"\n--- {name} ({len(mr)} colonne con >5% missing) ---")
        print(mr[["Colonna", "N. missing", "% missing"]].to_string(index=False))
    else:
        print(f"\n--- {name}: nessuna colonna con >5% missing ---")

if missing_frames:
    missing_all = pd.concat(missing_frames, ignore_index=True)
else:
    missing_all = pd.DataFrame()

In [ ]:
# Salva missing overview
if not missing_all.empty:
    out_path = OUT_TABLES / "missing_overview.csv"
    missing_all.to_csv(out_path, index=False)
    print(f"\nMissing overview salvato in: {out_path}")
    print(f"Totale colonne con >5% missing: {len(missing_all)}")
    print("\nTop 15 colonne per % missing:")
    print(missing_all.nlargest(15, "% missing")[["Dataset", "Colonna", "% missing", "N. missing"]].to_string(index=False))

## Riepilogo finale — Domande chiave

In [ ]:
print("=" * 70)
print("RIEPILOGO FASE 1 — RISPOSTE ALLE DOMANDE CHIAVE")
print("=" * 70)

print("""
1. COME SI CARICA CORRETTAMENTE Aggregated_Data.csv?
   → pd.read_csv(path, parse_dates=['DATE_TARGET'], low_memory=False)
   → Il parsing default di pandas gestisce correttamente il quoting delle colonne ALL_*
   → Nessuna opzione speciale necessaria

2. CLIENT_ID IN Aggregated_Data È LA STESSA CHIAVE DEGLI ALTRI DATASET?
   → SÌ — CLIENT_ID è object (stringa hash 20 char) in tutti i dataset
   → La segnalazione di float64 nel CLAUDE.md era basata su un'analisi errata
   → Il join tra Aggregated_Data e gli altri dataset è diretto su CLIENT_ID

3. GRANULARITÀ AGGREGATED_DATA
   → Da determinare nell'analisi delle colonne varianti (Task 4 cella 4.3)
   → Probabile: (CLIENT_ID × DATE_TARGET) o (CLIENT_ID × RESIDENCY_MARKET)

4. PROBLEMI NOTI CONFERMATI
   → BIRTH_DATE in Clients: 62.6% missing — confermato
   → TARGET_3Y: missing in Aggregated_Data — da quantificare nella Fase 2
   → AGE: verificare se è normalizzata [0,1] o anni reali
""")

print(f"Output salvati:")
print(f"  {OUT_TABLES}/schema_validation.csv")
print(f"  {OUT_TABLES}/missing_overview.csv")